In [267]:
%%capture
import pandas as pd
from pathlib import Path
import plotly.express as px

import torch
from nnsight import LanguageModel
from transformers import AutoTokenizer

dir = Path('../data/transcripts')
doc_paths = sorted(dir.glob('*.txt'))

# tokenize + collect activations
# id = 'allenai/Olmo-3-1125-32B'
id = 'Qwen/Qwen3-30B-A3B-Instruct-2507'
model = LanguageModel(id, device_map = 'auto', remote = 'true')
tokenizer = AutoTokenizer.from_pretrained(id)

In [280]:
with open(doc_paths[3], "r", encoding="utf-8") as f:
    michael = tokenizer.apply_chat_template([{'role': 'user', 'content': 'Think about this conversation and try your best to distinguish the people who are involved' + f.read()}], tokenize = False)

with open(doc_paths[4], "r", encoding="utf-8") as f:
    mike = tokenizer.apply_chat_template([{'role': 'user', 'content': 'Think about this conversation and try your best to distinguish the people who are involved' + f.read()}], tokenize = False)

In [282]:
# get tokens
michael_toks = tokenizer(michael, return_offsets_mapping = True)
mike_toks = tokenizer(mike, return_offsets_mapping = True)

def get_token_map(token_dict, string):
    token_map = {}
    token_map['token_id'] = token_dict['input_ids']
    token_map['offsets'] = token_dict['offset_mapping']
    token_map['token'] = [string[o[0]:o[1]] for o in token_map['offsets']]
    token_map['token_ix'] = [i for i in range(len(token_map['token']))]
    return pd.DataFrame(token_map)

michael_map = get_token_map(michael_toks, michael)
mike_map = get_token_map(mike_toks, mike)


In [308]:
# get hidden states
michael_hs = None
with model.trace(michael, remote = True):
    michael_hs = model.model.layers[24].output[0].save()

mike_hs = None
with model.trace(mike, remote = True):
    mike_hs = model.model.layers[24].output[0].save()


⬇ Downloading: 100%|██████████| 1.24M/1.24M [00:00<00:00]


⬇ Downloading: 100%|██████████| 1.24M/1.24M [00:00<00:00]


In [345]:
# now, we want a way to indicate which segments are "where the experiment is happening"
# in mech. interp you'll often have some special tokens you care about, and want to do tests on 
# and you want to be able to identify it dynamically (e.g. "which tokens correspond to a substring")
from utils.utils import flag_message_types
michael_map_labeled = flag_message_types(michael_map, base_messages = ['oh, well have fun at tennis, I think later we will go to the Fresh Market and get some pasta and salmon, goodbye!'])
mike_map_labeled = flag_message_types(mike_map, base_messages = ['oh, well have fun at tennis, I think later we will go to the Fresh Market and get some pasta and salmon, goodbye!'])


In [347]:
import numpy as np
import torch
from sklearn.decomposition import PCA

skip_n =  11
comb = torch.cat([michael_hs[skip_n:], mike_hs[skip_n:]], dim = 0).float().numpy()
pca = PCA(n_components = 2)
coords = pca.fit_transform(comb)

# important to check the shape (we know the pca should be the same for most elements)
michael_hs_pca = pd.DataFrame(coords[:michael_hs.shape[0] - skip_n, :])
mike_hs_pca = pd.DataFrame(coords[michael_hs.shape[0] - skip_n:, :])

# check 
print(michael_hs_pca, mike_hs_pca)

            0         1
0   -2.745486  0.742768
1   -1.701306 -0.432437
2   -1.508208 -2.081782
3   -0.352670 -3.243964
4   -1.564076  0.124485
..        ...       ...
369 -1.377362 -1.173215
370 -1.009165  0.551638
371 -3.590266 -2.116405
372 -2.760176 -3.198734
373 -3.716749 -2.751013

[374 rows x 2 columns]             0         1
0   -2.762740  0.740725
1   -1.705103 -0.432581
2   -1.488848 -2.155828
3   -0.364867 -3.190885
4   -1.582129  0.116560
..        ...       ...
369 -1.269093 -1.304812
370 -0.919226  0.445279
371 -3.449249 -2.433710
372 -2.701035 -3.294678
373 -3.566877 -2.728848

[374 rows x 2 columns]


In [348]:
# plotting 
michael_plot = pd.concat([michael_map_labeled.iloc[skip_n:].reset_index(drop = True), michael_hs_pca], axis = 1) # you need to reset the indice, so they'll try to join correctly (due to skip_n)
michael_plot['is_sequence'] = 'Michael' # the purpose of this is to identify the "Michael" one

mike_plot = pd.concat([mike_map_labeled.iloc[skip_n:].reset_index(drop = True), mike_hs_pca], axis = 1)
mike_plot['is_sequence'] = 'Mike'

In [349]:
michael_plot

,token_id,offsets,token,token_ix,base_message_ix,base_message,0,1,is_sequence
0,311,"(64, 67)",to,11,NaN,NaN,-2.745486,0.742768,Michael
1,32037,"(67, 79)",distinguish,12,NaN,NaN,-1.701306,-0.432437,Michael
2,279,"(79, 83)",the,13,NaN,NaN,-1.508208,-2.081782,Michael
3,1251,"(83, 90)",people,14,NaN,NaN,-0.352670,-3.243964,Michael
4,879,"(90, 94)",who,15,NaN,NaN,-1.564076,0.124485,Michael
...,...,...,...,...,...,...,...,...,...
369,11,"(1366, 1367)",",",380,0.0,"oh, well have fun at tennis, I think later we ...",-1.377362,-1.173215,Michael
370,46455,"(1367, 1375)",goodbye,381,0.0,"oh, well have fun at tennis, I think later we ...",-1.009165,0.551638,Michael
371,0,"(1375, 1376)",!,382,0.0,"oh, well have fun at tennis, I think later we ...",-3.590266,-2.116405,Michael
372,151645,"(1376, 1386)",<|im_end|>,383,NaN,NaN,-2.760176,-3.198734,Michael


In [350]:
plot_df = pd.concat([michael_plot, mike_plot], axis = 0)
plot_df["color"] = np.where(
    (plot_df["is_sequence"] == "Michael") & (plot_df["base_message_ix"] == 0.0),
    "Michael",
    np.where(
        (plot_df["is_sequence"] == "Mike") & (plot_df["base_message_ix"] == 0.0),
        "Mike",
        "Other"
    )
)

In [351]:
plot_df

,token_id,offsets,token,token_ix,base_message_ix,base_message,0,1,is_sequence,color
0,311,"(64, 67)",to,11,NaN,NaN,-2.745486,0.742768,Michael,Other
1,32037,"(67, 79)",distinguish,12,NaN,NaN,-1.701306,-0.432437,Michael,Other
2,279,"(79, 83)",the,13,NaN,NaN,-1.508208,-2.081782,Michael,Other
3,1251,"(83, 90)",people,14,NaN,NaN,-0.352670,-3.243964,Michael,Other
4,879,"(90, 94)",who,15,NaN,NaN,-1.564076,0.124485,Michael,Other
...,...,...,...,...,...,...,...,...,...,...
369,11,"(1363, 1364)",",",380,0.0,"oh, well have fun at tennis, I think later we ...",-1.269093,-1.304812,Mike,Mike
370,46455,"(1364, 1372)",goodbye,381,0.0,"oh, well have fun at tennis, I think later we ...",-0.919226,0.445279,Mike,Mike
371,0,"(1372, 1373)",!,382,0.0,"oh, well have fun at tennis, I think later we ...",-3.449249,-2.433710,Mike,Mike
372,151645,"(1373, 1383)",<|im_end|>,383,NaN,NaN,-2.701035,-3.294678,Mike,Other


In [352]:
import plotly.express as px
plot = px.scatter(plot_df, x = plot_df[0], y = plot_df[1], color = 'color', hover_data = ['token'])
plot

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'customdata': array([[' to'],
                                   [' distinguish'],
                                   [' the'],
                                   ...,
                                   [':'],
                                   ['<|im_end|>'],
                                   ['\n']], shape=(694, 1), dtype=object),
              'hovertemplate': 'color=Other<br>0=%{x}<br>1=%{y}<br>token=%{customdata[0]}<extra></extra>',
              'legendgroup': 'Other',
              'marker': {'color': '#636efa', 'symbol': 'circle'},
              'mode': 'markers',
              'name': 'Other',
              'orientation': 'v',
              'showlegend': True,
              'type': 'scatter',
              'x': {'bdata': ('DLYvwGPE2b/2DMG/K5G0vqEzyL/lhe' ... 'DAsw3AoQRSwDdwk8DA3SzAtkdkwA=='),
                    'dtype': 'f4'},
              'xaxis': 'x',
              'y': {'bdata': ('DyY+P2Vo3b7sOwXAGp1PwP7x/j3wNB' ... 'BvFfq/mZWDwDg/xb8A3FLAcaUuwA=='),
                    'dtype': 'f4'},
              'yaxis': 'y'},
             {'customdata': array([[' oh'],
                                   [','],
                                   [' well'],
                                   [' have'],
                                   [' fun'],
                                   [' at'],
                                   [' tennis'],
                                   [','],
                                   [' I'],
                                   [' think'],
                                   [' later'],
                                   [' we'],
                                   [' will'],
                                   [' go'],
                                   [' to'],
                                   [' the'],
                                   [' Fresh'],
                                   [' Market'],
                                   [' and'],
                                   [' get'],
                                   [' some'],
                                   [' pasta'],
                                   [' and'],
                                   [' salmon'],
                                   [','],
                                   [' goodbye'],
                                   ['!']], dtype=object),
              'hovertemplate': 'color=Michael<br>0=%{x}<br>1=%{y}<br>token=%{customdata[0]}<extra></extra>',
              'legendgroup': 'Michael',
              'marker': {'color': '#EF553B', 'symbol': 'circle'},
              'mode': 'markers',
              'name': 'Michael',
              'orientation': 'v',
              'showlegend': True,
              'type': 'scatter',
              'x': {'bdata': ('g4ONwM4hRcCPa3nAQkbLv8SELT8XRX' ... 'vTQGALoT+oSrxAak2wv1Esgb/sxmXA'),
                    'dtype': 'f4'},
              'xaxis': 'x',
              'y': {'bdata': ('pqQQPw9vzj/KWylAPWEKQHiOFUCWH+' ... 'Y7wDOKBD9w4nbA6SuWvyo4DT8tcwfA'),
                    'dtype': 'f4'},
              'yaxis': 'y'},
             {'customdata': array([[' oh'],
                                   [','],
                                   [' well'],
                                   [' have'],
                                   [' fun'],
                                   [' at'],
                                   [' tennis'],
                                   [','],
                                   [' I'],
                                   [' think'],
                                   [' later'],
                                   [' we'],
                                   [' will'],
                                   [' go'],
                                   [' to'],
                                   [' the'],
                                   [' Fresh'],
                                   [' Market'],
                                   [' and'],
                                   [' get'],
                                   [' some'],
                                

In [238]:
new_plot = px.scatter(plot_df, x = plot_df[0], y = plot_df[1], color = 'color', hover_data = ['token'])
new_plot

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'customdata': array([['Michael'],
                                   [':'],
                                   [' HE'],
                                   ...,
                                   ['.\n'],
                                   ['Mike'],
                                   [':']], shape=(676, 1), dtype=object),
              'hovertemplate': 'color=Other<br>0=%{x}<br>1=%{y}<br>token=%{customdata[0]}<extra></extra>',
              'legendgroup': 'Other',
              'marker': {'color': '#636efa', 'symbol': 'circle'},
              'mode': 'markers',
              'name': 'Other',
              'orientation': 'v',
              'showlegend': True,
              'type': 'scatter',
              'x': {'bdata': ('FJieRMfNZ8BrG3XA/nJywGPyccDgiX' ... 'BS92vA01NmwKPmYsAAyXjARUlkwA=='),
                    'dtype': 'f4'},
              'xaxis': 'x',
              'y': {'bdata': ('svi3PN+bDEG/as9AZVK/QM8JgUCtn6' ... '8YV4dAyBjjv3Ier0ChmtxAV9WJQA=='),
                    'dtype': 'f4'},
              'yaxis': 'y'},
             {'customdata': array([[' oh'],
                                   [','],
                                   [' well'],
                                   [' have'],
                                   [' fun'],
                                   [' at'],
                                   [' tennis'],
                                   [','],
                                   [' goodbye'],
                                   ['!']], dtype=object),
              'hovertemplate': 'color=Michael<br>0=%{x}<br>1=%{y}<br>token=%{customdata[0]}<extra></extra>',
              'legendgroup': 'Michael',
              'marker': {'color': '#EF553B', 'symbol': 'circle'},
              'mode': 'markers',
              'name': 'Michael',
              'orientation': 'v',
              'showlegend': True,
              'type': 'scatter',
              'x': {'bdata': 'hrN2wBZcb8Anv3DAGs59wHl7ccCV62rAEkpkwObaYcBT0H3AKhp0wA==', 'dtype': 'f4'},
              'xaxis': 'x',
              'y': {'bdata': 'iXeqQLMEdkAXUJlAlWE4QNI0Ob9CKAfANrAcwB6/RED2RxxA0fi8QA==', 'dtype': 'f4'},
              'yaxis': 'y'},
             {'customdata': array([[' oh'],
                                   [','],
                                   [' well'],
                                   [' have'],
                                   [' fun'],
                                   [' at'],
                                   [' tennis'],
                                   [','],
                                   [' goodbye'],
                                   ['!']], dtype=object),
              'hovertemplate': 'color=Mike<br>0=%{x}<br>1=%{y}<br>token=%{customdata[0]}<extra></extra>',
              'legendgroup': 'Mike',
              'marker': {'color': '#00cc96', 'symbol': 'circle'},
              'mode': 'markers',
              'name': 'Mike',
              'orientation': 'v',
              'showlegend': True,
              'type': 'scatter',
              'x': {'bdata': 'KR5owJ8sYMAsLmDAVtZ3wE53bcDofm/AT5FkwFGgX8BvvXnAYAlswA==', 'dtype': 'f4'},
              'xaxis': 'x',
              'y': {'bdata': 'quCaQE6nWkBF1YpA/n90QMRhYj7HLn6/Z5QiwCiGRkA86BhA8Wu2QA==', 'dtype': 'f4'},
              'yaxis': 'y'}],
    'layout': {'legend': {'title': {'text': 'color'}, 'tracegroupgap': 0},
               'margin': {'t': 60},
               'template': '...',
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': '0'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': '1'}}}
})

In [210]:
# this doesn't show much, but we expect the variance to be pretty small because it'll emphasize the 
# directions of highest variance (which could result from something else :))
plot

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'customdata': array([['Michael'],
                                   [':'],
                                   [' Hey'],
                                   ...,
                                   ['.\n'],
                                   ['Mike'],
                                   [':']], shape=(590, 1), dtype=object),
              'hovertemplate': 'color=Other<br>0=%{x}<br>1=%{y}<br>token=%{customdata[0]}<extra></extra>',
              'legendgroup': 'Other',
              'marker': {'color': '#636efa', 'symbol': 'circle'},
              'mode': 'markers',
              'name': 'Other',
              'orientation': 'v',
              'showlegend': True,
              'type': 'scatter',
              'x': {'bdata': ('50kdwSNnNMHxm5/BPViKwYRZxME9BK' ... 'E/+RaiQNO9HEGstr7BeBXRwVtsqMA='),
                    'dtype': 'f4'},
              'xaxis': 'x',
              'y': {'bdata': ('pooCwfMRLcAtgHtAHQjyQA7Gvz7Tdq' ... '5A3Pmtv0Epo0D0vYLBJlp3wcN5x0A='),
                    'dtype': 'f4'},
              'yaxis': 'y'},
             {'customdata': array([[' oh'],
                                   [','],
                                   [' well'],
                                   [' have'],
                                   [' fun'],
                                   [' at'],
                                   [' tennis'],
                                   [','],
                                   [' goodbye'],
                                   ['!']], dtype=object),
              'hovertemplate': 'color=Michael<br>0=%{x}<br>1=%{y}<br>token=%{customdata[0]}<extra></extra>',
              'legendgroup': 'Michael',
              'marker': {'color': '#EF553B', 'symbol': 'circle'},
              'mode': 'markers',
              'name': 'Michael',
              'orientation': 'v',
              'showlegend': True,
              'type': 'scatter',
              'x': {'bdata': '94q3wCgwvsDotHTArf1swEHarz/x4RpA1QZIQJI4xL9oddTAYyFdwQ==', 'dtype': 'f4'},
              'xaxis': 'x',
              'y': {'bdata': 'rRnxQKBq+EDI1ARBetoqQQN1Y0GlTJdAC0HxQPoAJkE2kD5BgWkxPw==', 'dtype': 'f4'},
              'yaxis': 'y'},
             {'customdata': array([[' oh'],
                                   [','],
                                   [' well'],
                                   [' have'],
                                   [' fun'],
                                   [' at'],
                                   [' tennis'],
                                   [','],
                                   [' goodbye'],
                                   ['!']], dtype=object),
              'hovertemplate': 'color=Mike<br>0=%{x}<br>1=%{y}<br>token=%{customdata[0]}<extra></extra>',
              'legendgroup': 'Mike',
              'marker': {'color': '#00cc96', 'symbol': 'circle'},
              'mode': 'markers',
              'name': 'Mike',
              'orientation': 'v',
              'showlegend': True,
              'type': 'scatter',
              'x': {'bdata': 'm3CJwHlA7r8CYjbA04TKv7eHoT7DGJ++eW4YQCKjScAjadfA/HhHwQ==', 'dtype': 'f4'},
              'xaxis': 'x',
              'y': {'bdata': 'R0nUQMP5rUAa/MxAG6v2QDIscEHzTQJBNUQKQc1dMUGNTERBxnRXPw==', 'dtype': 'f4'},
              'yaxis': 'y'}],
    'layout': {'legend': {'title': {'text': 'color'}, 'tracegroupgap': 0},
               'margin': {'t': 60},
               'template': '...',
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': '0'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': '1'}}}
})